In [1]:
import hopsworks
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

# ── Connect ────────────────────────────────────────────────────────────────────
project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST", "eu-west.cloud.hopsworks.ai"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs      = project.get_feature_store()
fg      = fs.get_feature_group(name="aqi_features", version=1)
df      = fg.read(online=False)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df      = df.sort_values("timestamp").reset_index(drop=True)

print(f"Total rows:  {len(df)}")
print(f"Date range:  {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Columns:     {len(df.columns)}")

# ── Basic stats ────────────────────────────────────────────────────────────────
print("\n--- AQI stats ---")
print(df["aqi"].describe().round(2))

# ── Check forecast column coverage ────────────────────────────────────────────
fc_cols = [c for c in df.columns if c.startswith("fc_")]
print(f"\n--- Forecast columns ({len(fc_cols)}) ---")
print(df[fc_cols].notna().sum().to_string())

# ── Rows WITH forecast features ───────────────────────────────────────────────
rows_with_fc = df[df["fc_pm25_24h"].notna()]
print(f"\n--- Rows with forecast features: {len(rows_with_fc)} ---")
print(rows_with_fc[["timestamp", "aqi", "fc_pm25_24h", "fc_dust_48h", "fc_uvi_72h"]].tail(10).to_string())

# ── Rows with real AQI (training eligible) ────────────────────────────────────
real_aqi = df[df["aqi"].notna()]
print(f"\n--- Rows with real AQI (training eligible): {len(real_aqi)} ---")
print(real_aqi[["timestamp", "aqi", "pm25", "pm10", "rolling_mean_24h"]].tail(10).to_string())

# ── Check NaN counts per column ───────────────────────────────────────────────
print("\n--- NaN counts per column ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string())

# ── AQI distribution by year ───────────────────────────────────────────────────
print("\n--- AQI mean by year ---")
print(df.groupby(df["timestamp"].dt.year)["aqi"].agg(["mean", "count", "min", "max"]).round(2))

# ── Latest 5 rows ─────────────────────────────────────────────────────────────
print("\n--- Latest 5 rows ---")
print(df.tail(5)[["timestamp", "aqi", "pm25", "pm10", "aqi_lag_1h", 
                   "rolling_mean_24h", "fc_pm25_24h", "fc_dust_48h"]].to_string())

# ── Earliest 5 rows ───────────────────────────────────────────────────────────
print("\n--- Earliest 5 rows ---")
print(df.head(5)[["timestamp", "aqi", "pm25", "pm10", "aqi_lag_1h",
                   "rolling_mean_24h", "fc_pm25_24h"]].to_string())

d:\Projects\Intern Project\AQI_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-15 12:40:46,785 INFO: Initializing external client
2026-05-15 12:40:46,785 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-15 12:40:49,548 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.94s) 
Total rows:  2340
Date range:  2026-01-29 00:00:00+00:00 → 2026-05-15 06:00:00+00:00
Columns:     48

--- AQI stats ---
count    2340.00
mean       82.78
std        24.08
min        14.58
25%        66.56
50%        78.34
75%        93.90
max       163.60
Name: aqi, dtype: float64

--- Forecast columns (21) ---
fc_pm25_24h    2339
fc_co_24h      2339
fc_no2_24h     2339
fc_so2_24h     2339
fc_o3_24h      2339
fc_dust_24h    2339
fc_uvi_24h     2339
fc_pm25_48h    2339
fc_co_48h      2339
fc_no2_48h     2339
fc_so2_48h     2339
fc_o3_48h      2339
fc_dust_48h    2339
fc_uvi_48h     2339
fc_pm25_72h    2339
fc_co_72h      2339


In [1]:
import hopsworks, os
from dotenv import load_dotenv
load_dotenv()

project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs = project.get_feature_store()
fg = fs.get_feature_group("aqi_features", version=1)
df = fg.read(online=False)
print("Total rows:", len(df))
print("Latest timestamp:", df["timestamp"].max())
print("Oldest timestamp:", df["timestamp"].min())

d:\Projects\Intern Project\AQI_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-08 16:32:20,833 INFO: Initializing external client
2026-05-08 16:32:20,833 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-08 16:32:23,540 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (21.76s) 
Total rows: 2259
Latest timestamp: 2026-05-03 14:00:00+00:00
Oldest timestamp: 2026-01-29 00:00:00+00:00


In [2]:

import hopsworks, os
from dotenv import load_dotenv
load_dotenv()
project = hopsworks.login(host=os.getenv('HOPSWORKS_HOST'), api_key_value=os.getenv('HOPSWORKS_API_KEY'))
fs = project.get_feature_store()
fg = fs.get_feature_group('aqi_features', version=1)
import pandas as pd
df = fg.read(online=False)
print('Total rows:', len(df))
print('Latest timestamp:', df['timestamp'].max())

2026-05-08 17:20:55,206 INFO: Closing external client and cleaning up certificates.
2026-05-08 17:20:55,214 INFO: Connection closed.
2026-05-08 17:20:55,216 INFO: Initializing external client
2026-05-08 17:20:55,217 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-08 17:20:57,739 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.82s) 
Total rows: 2261
Latest timestamp: 2026-05-08 10:00:00+00:00


In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
import hopsworks
from aqi_feature_utils import pm25_to_aqi

load_dotenv()
project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST", "eu-west.cloud.hopsworks.ai"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs = project.get_feature_store()
fg = fs.get_feature_group('aqi_features', version=1)
df = fg.read(online=False)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# ── Find MAX values ────────────────────────────────────────────────────────────
print(f"MAX PM2.5: {df['pm25'].max():.1f}")
print(f"MAX stored AQI: {df['aqi'].max():.1f}")

# ── Find rows with highest PM2.5 ────────────────────────────────────────────────
top_pm25 = df.nlargest(5, 'pm25')[['timestamp', 'pm25', 'aqi']]
top_pm25['aqi_calc'] = top_pm25['pm25'].apply(pm25_to_aqi)
print("\n--- Top 5 PM2.5 rows ---")
print(top_pm25.to_string())

# ── Find rows with highest AQI ────────────────────────────────────────────────
top_aqi = df.nlargest(5, 'aqi')[['timestamp', 'pm25', 'aqi']]
top_aqi['aqi_calc'] = top_aqi['pm25'].apply(pm25_to_aqi)
print("\n--- Top 5 AQI rows ---")
print(top_aqi.to_string())


2026-05-15 12:45:49,028 INFO: Closing external client and cleaning up certificates.
2026-05-15 12:45:49,031 INFO: Connection closed.
2026-05-15 12:45:49,033 INFO: Initializing external client
2026-05-15 12:45:49,034 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-15 12:45:51,534 INFO: Python Engine initialized.
